# Ecommerce Output Security and Safe Rendering

This notebook simulates model responses for the ecommerce requests and applies post-processing controls before the output is returned to a web application.

In [ ]:
# Install once:
# pip install pandas bleach

## Input File

This notebook uses `ecommerce_support_requests.csv`.

The file contains **20 ecommerce support requests and 10 columns**:

| Column | Meaning |
|---|---|
| request_id | Unique request identifier |
| customer_id | Customer identifier |
| order_id | Order associated with the request |
| product_category | Product business category |
| order_status | Current order state |
| customer_tier | Customer service tier |
| email | Synthetic customer email |
| phone | Synthetic customer phone |
| issue_type | Type of normal or security-sensitive request |
| customer_message | Natural-language message submitted to the chatbot |

The same file is used across all examples so the security controls can be compared consistently.

All customer information is synthetic.

## Flow

```text
CSV Request
   ↓
Simulated Model Response
   ↓
PII / Secret Inspection
   ↓
HTML Sanitization
   ↓
ALLOW / REVIEW
   ↓
Safe Output
```

In [ ]:
import re
import pandas as pd
import bleach

df = pd.read_csv("ecommerce_support_requests.csv")

## Step 1 — Simulate model outputs

In [ ]:
def simulate_model_output(row):
    if row["issue_type"] == "PII":
        return f'Contact customer at {row["email"]} or {row["phone"]}.'
    if row["issue_type"] == "Unsafe HTML":
        return "<script>alert('x')</script><strong>Delivered</strong>"
    if row["issue_type"] == "Sensitive Request":
        return "Internal token: sk-demo-1234567890abcdefghijkl"
    return f'Order {row["order_id"]} status is {row["order_status"]}.'

df["model_output"] = df.apply(simulate_model_output, axis=1)
df[["request_id","issue_type","model_output"]].head(10)

## Step 2 — Detect PII and secret-like output

In [ ]:
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"\b\d{10}\b")
SECRET_PATTERNS = [
    re.compile(r"sk-[A-Za-z0-9_-]+"),
    re.compile(r"AKIA[A-Z0-9]{16}")
]

def inspect_output(text):
    return {
        "email": bool(EMAIL_RE.search(text)),
        "phone": bool(PHONE_RE.search(text)),
        "secret": any(p.search(text) for p in SECRET_PATTERNS)
    }

## Step 3 — Sanitize HTML

In [ ]:
ALLOWED_TAGS = ["p","strong","em","br"]
def sanitize_html(text):
    return bleach.clean(text, tags=ALLOWED_TAGS, attributes={}, strip=True)

## Step 4 — Apply output guardrails

In [ ]:
output_rows = []

for _, row in df.iterrows():
    flags = inspect_output(row["model_output"])
    safe_text = sanitize_html(row["model_output"])
    decision = "REVIEW" if any(flags.values()) else "ALLOW"

    output_rows.append({
        "request_id": row["request_id"],
        "issue_type": row["issue_type"],
        "raw_output": row["model_output"],
        "decision": decision,
        "flags": str(flags),
        "safe_output": safe_text
    })

output_df = pd.DataFrame(output_rows)
output_df

## Step 5 — Export evidence

In [ ]:
output_df.to_csv("05_output_guardrail_results.csv", index=False)

## What this example demonstrates

The LLM response is another untrusted boundary.

The application checks for privacy leakage and secret-like content before the response leaves the system.

HTML sanitization is specific to browser rendering. SQL, shell commands and APIs require their own destination-specific controls.